# ETL Pipeline Demo — Bibliometrix Python
## Advanced Level 

This notebook demonstrates the ETL pipeline developed for the Bibliometrix-Python project.
The pipeline extracts data from OpenAlex and PubMed APIs, transforms it into the WoS standard schema, and validates the output.

---
## PHASE 1: EXTRACT
Data is retrieved via REST APIs from OpenAlex and PubMed.
The `retrieve()` function handles pagination, rate limits, and retries automatically.

In [5]:
from www.services.api_retriever import retrieve

print("=== EXTRACT: OpenAlex ===")
records_oa = retrieve(query="machine learning", platform="openalex", total=10)
print(f"Records retrieved: {len(records_oa)}")
print(f"Sample raw keys: {list(records_oa[0].keys())[:8]}")
print(f"\nSample title: {records_oa[0].get('title', 'N/A')}")

=== EXTRACT: OpenAlex ===
Records retrieved: 10
Sample raw keys: ['id', 'doi', 'title', 'display_name', 'relevance_score', 'publication_year', 'publication_date', 'ids']

Sample title: Scikit-learn: Machine Learning in Python


In [6]:
print("=== EXTRACT: PubMed ===")
records_pm = retrieve(query="machine learning", platform="pubmed", total=10)
print(f"Records retrieved: {len(records_pm)}")
print(f"Sample raw keys: {list(records_pm[0].keys())[:8]}")
print(f"\nSample title: {records_pm[0].get('Title', 'N/A')}")

=== EXTRACT: PubMed ===
ICITE RAW PARAMS: {'pmids': '42312418,42312373,42312342,42312334,42312331,42312232,42312183,42312129,42312085,42312038', 'fl': 'pmid,citation_count'}
ICITE RAW RESPONSE TYPE: <class 'dict'>
ICITE RAW RESPONSE SAMPLE: {'data': []}
ICITE DEBUG: 10 PMIDs sent, 0 matched. Sample: []
Records retrieved: 10
Sample raw keys: ['uid', 'pubdate', 'epubdate', 'source', 'authors', 'lastauthor', 'title', 'sorttitle']

Sample title: N/A


---
## PHASE 2: TRANSFORM
Raw API responses are mapped to the WoS standard schema using mapping dictionaries.
Multi-value fields are cast to `list[str]`, scalar fields to `str`, and `TC` to `int`.

In [8]:
from www.services.standardizer import standardize
import pandas as pd

print("=== TRANSFORM: OpenAlex ===")
df_oa = standardize(records_oa, source="openalex")
print(f"Shape: {df_oa.shape}")
print(f"Columns: {df_oa.columns.tolist()}")
df_oa[['AU', 'TI', 'PY', 'SO', 'TC', 'DB']].head(3)

=== TRANSFORM: OpenAlex ===
Shape: (10, 25)
Columns: ['UT', 'DI', 'TI', 'PY', 'LA', 'DT', 'TC', 'SO', 'JI', 'AU', 'AF', 'C1', 'RP', 'AB', 'VL', 'IS', 'BP', 'EP', 'DE', 'CR', 'ID', 'PMID', 'DB', 'SR', 'SR_FULL']


,AU,TI,PY,SO,TC,DB
0,"[Fabián Pedregosa, Gaël Varoquaux, Alexandre G...",Scikit-learn: Machine Learning in Python,2012,ARXIV (CORNELL UNIVERSITY),63730,OPENALEX
1,[],"Genetic algorithms in search, optimization, an...",1989,CHOICE REVIEWS ONLINE,49334,OPENALEX
2,[J. R. Quinlan],C4.5: Programs for Machine Learning,1992,,23698,OPENALEX


In [9]:
print("=== TRANSFORM: PubMed ===")
df_pm = standardize(records_pm, source="pubmed")
print(f"Shape: {df_pm.shape}")
print(f"Columns: {df_pm.columns.tolist()}")
df_pm[['AU', 'TI', 'PY', 'SO', 'TC', 'DB']].head(3)

=== TRANSFORM: PubMed ===
Shape: (10, 26)
Columns: ['UT', 'TI', 'SO', 'JI', 'PY', 'VL', 'IS', 'LA', 'DT', 'RP', 'AU', 'AF', 'DI', 'PMID', 'BP', 'EP', 'CR', 'AB', 'C1', 'DE', 'ID', 'AU_CO', 'TC', 'DB', 'SR', 'SR_FULL']


,AU,TI,PY,SO,TC,DB
0,"[Al-Qerem W, Jarab A, Eberhardt J, Abdo S, Al-...","Health literacy, medication adherence, and qua...",2026,The Libyan journal of medicine,0,PUBMED
1,"[Stärk P, Stooß H, Langer MF, Rumiantsev E, Sc...",Simultaneous learning of static and dynamic ch...,2026,Physical chemistry chemical physics : PCCP,0,PUBMED
2,"[Selitser M, McWhinney SR, Wu L, Fraiha-Pegado...",The role of metabolic health in neurostructura...,2026,Psychological medicine,0,PUBMED


### Inspect multi-value fields
Author keywords (`DE`) and cited references (`CR`) must be `list[str]`.

In [11]:
print("=== Multi-value fields (OpenAlex) ===")
print(f"AU type: {type(df_oa['AU'].iloc[0])}")
print(f"AU sample: {df_oa['AU'].iloc[0]}")
print(f"\nDE type: {type(df_oa['DE'].iloc[0])}")
print(f"DE sample: {df_oa['DE'].iloc[0]}")
print(f"\nCR type: {type(df_oa['CR'].iloc[0])}")
print(f"CR sample (first 2): {df_oa['CR'].iloc[0][:2]}")

=== Multi-value fields (OpenAlex) ===
AU type: <class 'list'>
AU sample: ['Fabián Pedregosa', 'Gaël Varoquaux', 'Alexandre Gramfort', 'Vincent Michel', 'Bertrand Thirion', 'Olivier Grisel', 'Mathieu Blondel', 'Müller, Andreas', 'Nothman, Joel', 'Louppe, Gilles', 'Peter Prettenhofer', 'Ron J. Weiss', 'Vincent Dubourg', 'Jake Vanderplas', 'Alexandre Passos', 'David Cournapeau', 'Matthieu Brucher', 'Matthieu Perrot', 'Édouard Duchesnay']

DE type: <class 'list'>
DE sample: ['Python (programming language)', 'Documentation', 'Computer science', 'MIT License', 'Artificial intelligence', 'Machine learning', 'Programming language', 'License', 'Software engineering', 'Operating system']

CR type: <class 'list'>
CR sample (first 2): ['https://openalex.org/W1496508106', 'https://openalex.org/W1571024744']


---
## PHASE 3: VALIDATE
The validation module checks:
1. All mandatory columns exist
2. No NaN or None values remain
3. Multi-value columns are correctly typed as lists

In [13]:
from www.services.validator import validate

print("=== VALIDATE: OpenAlex ===")
df_oa = validate(df_oa)
print(f"\nSR sample: {df_oa['SR'].iloc[0]}")
df_oa[['SR', 'DE', 'AB']].head(3)

=== VALIDATE: OpenAlex ===
Running validation...
  PASS — all mandatory columns present
  PASS — no null values found
  PASS — all column types correct
Validation passed.

SR sample: Fabián Pedregosa, 2012, arXiv (Cornell University)


,SR,DE,AB
0,"Fabián Pedregosa, 2012, arXiv (Cornell Univers...","[Python (programming language), Documentation,...",Scikit-learn is a Python module integrating a ...
1,"NA, 1989, Choice Reviews Online","[Computer science, Artificial intelligence, Ma...",From the Publisher:\r\nThis book brings togeth...
2,"J. R. Quinlan, 1992,","[Computer science, Unix, Classifier (UML), Mac...",Classifier systems play a major role in machin...


In [14]:
print("=== VALIDATE: PubMed ===")
df_pm = validate(df_pm)
print(f"\nSR sample: {df_pm['SR'].iloc[0]}")
df_pm[['SR', 'DE', 'AB']].head(3)

=== VALIDATE: PubMed ===
Running validation...
  PASS — all mandatory columns present
  PASS — no null values found
  PASS — all column types correct
Validation passed.

SR sample: Al-Qerem W, 2026, Libyan J Med


,SR,DE,AB
0,"Al-Qerem W, 2026, Libyan J Med","[Algorithms, health care, medication adherence...",AIMS: This study examined the association of h...
1,"Stärk P, 2026, Phys Chem Chem Phys",[],Long-range interactions and electric response ...
2,"Selitser M, 2026, Psychol Med","[BrainAGE, bipolar disorders, brain structure,...",BACKGROUND: Bipolar disorders (BD) rank among ...


---
## FULL PIPELINE — 200 records
End-to-end demonstration with 200 records per platform, exported to CSV.

In [16]:
print("=== FULL PIPELINE: OpenAlex (200 records) ===")
records_oa_200 = retrieve(query="machine learning", platform="openalex", total=200)
df_oa_200 = standardize(records_oa_200, source="openalex")
df_oa_200 = validate(df_oa_200)
df_oa_200.to_csv("test_openalex_200.csv", index=False)
print(f"Shape: {df_oa_200.shape}")
print("CSV saved: test_openalex_200.csv")
df_oa_200[['AU', 'TI', 'PY', 'SO', 'TC', 'SR']].head(10)

=== FULL PIPELINE: OpenAlex (200 records) ===
Running validation...
  PASS — all mandatory columns present
  PASS — no null values found
  PASS — all column types correct
Validation passed.
Shape: (200, 25)
CSV saved: test_openalex_200.csv


,AU,TI,PY,SO,TC,SR
0,"[Fabián Pedregosa, Gaël Varoquaux, Alexandre G...",Scikit-learn: Machine Learning in Python,2012,ARXIV (CORNELL UNIVERSITY),63730,"Fabián Pedregosa, 2012, arXiv (Cornell Univers..."
1,[],"Genetic algorithms in search, optimization, an...",1989,CHOICE REVIEWS ONLINE,49334,"NA, 1989, Choice Reviews Online"
2,[J. R. Quinlan],C4.5: Programs for Machine Learning,1992,,23698,"J. R. Quinlan, 1992,"
3,[Arthur Asuncion],UCI Machine Learning Repository,2007,MEDICAL ENTOMOLOGY AND ZOOLOGY,24350,"Arthur Asuncion, 2007, Medical Entomology and ..."
4,"[Ian H. Witten, Eibe Frank, Mark A. Hall]",Data Mining: Practical Machine Learning Tools ...,2011,ELSEVIER EBOOKS,25713,"Ian H. Witten, 2011, Elsevier eBooks"
5,[Nasser M. Nasrabadi],Pattern Recognition and Machine Learning,2007,JOURNAL OF ELECTRONIC IMAGING,22083,"Nasser M. Nasrabadi, 2007, Journal of Electron..."
6,[David E. Goldberg],"Genetic Algorithms in Search, Optimization and...",1988,,17771,"David E. Goldberg, 1988,"
7,[],Proceedings of the 24th international conferen...,2007,,11734,"NA, 2007,"
8,"[Carl Edward Rasmussen, Christopher K. I. Will...",Gaussian Processes for Machine Learning,2005,THE MIT PRESS EBOOKS,10489,"Carl Edward Rasmussen, 2005, The MIT Press eBooks"
9,[Kevin P. Murphy],Machine learning a probabilistic perspective,2012,,9328,"Kevin P. Murphy, 2012,"


In [17]:
print("=== FULL PIPELINE: PubMed (200 records) ===")
records_pm_200 = retrieve(query="machine learning AND 2015:2018[dp]", platform="pubmed", total=200)
df_pm_200 = standardize(records_pm_200, source="pubmed")
df_pm_200 = validate(df_pm_200)
df_pm_200.to_csv("test_pubmed_200.csv", index=False)
print(f"Shape: {df_pm_200.shape}")
print("CSV saved: test_pubmed_200.csv")
df_pm_200[['AU', 'TI', 'PY', 'SO', 'TC', 'SR']].head(10)

=== FULL PIPELINE: PubMed (200 records) ===
ICITE RAW PARAMS: {'pmids': '35498556,33840866,32489521,32071505,31885411,31839683,31768504,31728432,31639740,31595179,31571703,33748314,33312073,31533900,31533899,31518298,31496926,31458929,31458044,31457027,31447541,31406291,31404438,31404415,31360877,31354187,31338374,31334499,31330270,31312809,31304341,31304340,31304339,31304336,31304333,31304331,31304329,31304318,31304314,31274357,31263802,31258180,33134651,31245585,31240237,31218278,31215903,31208691,31191959,31179446,31162468,31160830,31131381,31128628,31111116,31108778,31106304,31106302,31096424,31093008,31065411,31061990,31061817,31057954,31031415,31021206,31015728,31015720,31015713,31015651,31015650,31015648,31015647,31015645,31014789,31014788,31014679,31012547,31011686,31008043,31001412,31001455,30976758,30976397,30956748,30949431,30948806,30931438,30931437,30921746,30919393,30919392,30906841,30906832,30906397,30906113,30899763,30896099,30895278,30886441,30881694,30881655,30875708,

,AU,TI,PY,SO,TC,SR
0,"[Yue C, Ware S, Morillo R, Lu J, Shang C, Bi J...",Fusing Location Data for Depression Prediction.,2021,IEEE transactions on big data,13,"Yue C, 2021, IEEE Trans Big Data"
1,"[Fu Q, Guo X, Land KC]",Optimizing Count Responses in Surveys: A Machi...,2020,Sociological methods & research,4,"Fu Q, 2020, Sociol Methods Res"
2,"[Taylor S, Jaques N, Nosakhare E, Sano A, Pica...",Personalized Multitask Learning for Predicting...,2020,IEEE transactions on affective computing,54,"Taylor S, 2020, IEEE Trans Affect Comput"
3,"[Mohan IK, Khan SA, Shiva Krishna D, Vijaya Bh...",Adaptive Neuro-Fuzzy Inference System-Based Ex...,2020,Indian journal of clinical biochemistry : IJCB,0,"Mohan IK, 2020, Indian J Clin Biochem"
4,"[Wang X, Zhao K, Cha S, Amato MS, Cohn AM, Pea...",Mining User-Generated Content in an Online Smo...,2019,Decision support systems,11,"Wang X, 2019, Decis Support Syst"
5,"[Hall EC, Raskutti G, Willett RM]",Learning High-dimensional Generalized Linear A...,2019,IEEE transactions on information theory,3,"Hall EC, 2019, IEEE Trans Inf Theory"
6,"[Dash D, Ferrari P, Malik S, Montillo A, Maldj...",Determining the Optimal Number of MEG Trials: ...,2019,"Brain informatics : international conference, ...",7,"Dash D, 2019, Brain Inform (2018)"
7,"[Shen F, Larson DW, Naessens JM, Habermann EB,...",Detection of Surgical Site Infection Utilizing...,2019,Journal of healthcare informatics research,15,"Shen F, 2019, J Healthc Inform Res"
8,"[Al Jalbout N, Troncoso R Jr, Evans JD, Rothma...",Biomarkers and Molecular Diagnostics for Early...,2019,The journal of applied laboratory medicine,12,"Al Jalbout N, 2019, J Appl Lab Med"
9,"[Imtiaz H, Sarwate AD]",Distributed Differentially-Private Algorithms ...,2018,IEEE journal of selected topics in signal proc...,5,"Imtiaz H, 2018, IEEE J Sel Top Signal Process"


---
## Summary

| Platform | Records | Columns | NaN | SR |
|----------|---------|---------|-----|----|
| OpenAlex | 200 | 26 | 0 | ✅ |
| PubMed   | 200 | 26 | 0 | ✅ |

The ETL pipeline successfully:
- Extracted data from OpenAlex and PubMed REST APIs
- Transformed raw JSON into the WoS standard schema
- Enforced type contracts (list[str], str, int)
- Validated all mandatory columns
- Generated standardized CSV files ready for Bibliometrix-Python analysis